# Combining Data

*Stack compatible row batches and attach information using a shared key.*


## Stack Rows with `concat`

`pd.concat([first, second], ignore_index=True)` appends the rows of two
compatible tables and assigns consecutive index labels. Columns match by
name. These are two batches of student scores with the same schema.


In [1]:
import pandas as pd

first_batch = pd.DataFrame({"name": ["Minji", "Joon"], "score": [82, 91]})
second_batch = pd.DataFrame({"name": ["Hana", "Taeho"], "score": [95, 87]})
combined_df = pd.concat([first_batch, second_batch], ignore_index=True)

print("Combined scores:")
print(combined_df)


Combined scores:
    name  score
0  Minji     82
1   Joon     91
2   Hana     95
3  Taeho     87


The result has four rows with index labels 0 through 3. Concatenation adds records; it does not look up extra fields for a student.


## Attach Columns with `merge`

`pd.merge(left, right, on="name", how="left")` matches the shared key
and keeps all left-side records. These prepared names uniquely identify
students; a real roster should use stable student IDs.
`validate="many_to_one"` checks that the right-hand key is unique, so
repeated lookup keys cannot unexpectedly multiply rows.


In [2]:
student_info = pd.DataFrame({
    "name": ["Minji", "Joon", "Hana", "Taeho"],
    "major": ["Data Science", "AI", "Economics", "Design"],
})
merged_df = pd.merge(
    combined_df, student_info, on="name", how="left", validate="many_to_one"
)

print("Scores with student information:")
print(merged_df)
print("Row count:", len(merged_df))


Scores with student information:
    name  score         major
0  Minji     82  Data Science
1   Joon     91            AI
2   Hana     95     Economics
3  Taeho     87        Design
Row count: 4


Each of the four scores receives the matching major. Row order does not need to match between the input tables because the merge uses names.


## Decide What Happens to Unmatched Keys

An inner merge keeps only matched keys. A left merge retains unmatched
left records and uses missing values for unavailable right-side fields.
Reuse the roster with Taeho's information temporarily absent.


In [3]:
available_info = student_info.loc[student_info["name"] != "Taeho"]
left_result = pd.merge(
    combined_df, available_info, on="name", how="left", validate="many_to_one"
)
inner_result = pd.merge(
    combined_df, available_info, on="name", how="inner", validate="many_to_one"
)

print("Left merge:")
print(left_result)
print("Inner merge names:", inner_result["name"].tolist())
print("Left and inner row counts:", len(left_result), len(inner_result))


Left merge:
    name  score         major
0  Minji     82  Data Science
1   Joon     91            AI
2   Hana     95     Economics
3  Taeho     87           NaN
Inner merge names: ['Minji', 'Joon', 'Hana']
Left and inner row counts: 4 3


The left result keeps four students, with a missing major for Taeho. The inner result has only Minji, Joon, and Hana.


Use `concat` to add compatible rows and `merge` to attach related columns. Check key uniqueness and choose how unmatched records should be handled.

[Combining pandas objects](https://pandas.pydata.org/docs/user_guide/merging.html)
